In [14]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph,START,END 
from dotenv import load_dotenv
import os
from pydantic import BaseModel,Field
from typing import TypedDict

In [15]:
load_dotenv()


AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

In [ ]:
class generate_structured_joke_output(BaseModel):
    joke:str = Field(description="the joke")
    explanation:str = Field(description="the explanation of the joke")
class Gen_joke_input(TypedDict):
    topic: str
    joke: str
    explanation: str
    
structured_model = model.with_structured_output(generate_structured_joke_output)

In [17]:
def gen_joke(state:Gen_joke_input):
    prompt = f"Generate a joke about {state['topic']} and explain it in detail"
    response = structured_model.invoke(prompt)
    return {
        'joke': response.joke,
        'explanation': response.explanation
    }

In [18]:
graph = StateGraph(Gen_joke_input)
graph.add_node("gen_joke",gen_joke)

graph.add_edge(START,"gen_joke")
graph.add_edge("gen_joke",END)

workflow = graph.compile()
initial_state = {
    'topic':"pizza"
}
result = workflow.invoke(initial_state)

print(result)

{'topic': 'pizza', 'joke': 'Why did the pizza apply for a job? Because it needed the dough.', 'explanation': "This joke is funny because it uses a double meaning of the word 'dough.' In the context of pizza, dough is the raw mixture of flour, water, and other ingredients used to make the crust. In the context of jobs and money, 'dough' is also a slang term for money. So the pizza applying for a job 'because it needed the dough' humorously connects the idea of pizza literally being made from dough with the human motivation of working to earn money."}


Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 16422
API Key: lsv2_********************************************1a
post: trace=0aff77e2-c942-4dad-b2e4-1c3e391a2ffc,id=e9e08efa-9966-4f0a-98cd-2c53a02f4aa6; patch: trace=0aff77e2-c942-4dad-b2e4-1c3e391a2ffc,id=0aff77e2-c942-4dad-b2e4-1c3e391a2ffc; trace=0aff77e2-c942-4dad-b2e4-1c3e391a2ffc,id=3901d2f2-e94c-471a-ac9e-1632a792e840; trace=0aff77e2-c942-4dad-b2e4-1c3e391a2ffc,id=b04d9cf2-7cd0-4693-aa16-0c795eab986a; trace=0aff77e2-c942-4dad-b2e4-1c3e391a2ffc,id=917b73f1-d